Paper equations 4–7: strongest dummy gives one unknown logit. beta=1, gamma=0.1. Reference detection uses probability difference at T=1024; bias is fit on known validation only.

In [ ]:
def augmented_logits(z,d):
    return torch.cat([z,d.max(dim=1,keepdim=True).values],dim=1)

def proser_loss(model,x,y):
    if len(y)%2:raise ValueError('PROSER requires even minibatches.')
    half=len(y)//2
    _,z,d=forward_outputs(model,x[:half])
    a=augmented_logits(z,d)
    ce=F.cross_entropy(a,y[:half])
    masked=a.scatter(1,y[:half,None],float('-inf'))
    cp=F.cross_entropy(masked,torch.full_like(y[:half],10))
    h=pre_features(model,x[half:])
    mixed,pairs=mix_hidden(h,y[half:])
    dp=a.sum()*0.
    if pairs:
        f=post_features(model,mixed)
        mix_logits=augmented_logits(model.fc(f),model.dummy(f))
        dp=F.cross_entropy(mix_logits,torch.full((pairs,),10,dtype=torch.long,device=x.device))
    loss=ce+cp+.1*dp
    return loss,dict(ce=ce.detach().item(),classifier_placeholder=cp.detach().item(),data_placeholder=dp.detach().item(),pairs=pairs,correct=(z.argmax(1)==y[:half]).sum().item(),count=half)

def fit_placeholder_bias(z,d):
    return float(np.quantile(z.max(1)-d.max(1),.05,method='linear'))

def placeholder_score(z,d,bias):
    a=np.concatenate([z,d.max(1,keepdims=True)+bias],axis=1).astype(np.float64)/1024.
    a-=a.max(1,keepdims=True)
    p=np.exp(a);p/=p.sum(1,keepdims=True)
    return p[:,10]-p[:,:10].max(1)